Lecture: AI I - Advanced 

Previous:
[**Chapter 1.1: Neuron**](../01_neuron.ipynb)

---

# Exercise 1.1: Neuron

> Hint: When doing the exercises put your solution in the designated "Solution" section:
> ```python
> # Solution (put your code here)
> ```

## Task 1: Implement Activation Functions from Scratch

Implement the core activation functions without using PyTorch's built-in functions.

**Tasks**:
- Implement ReLU, Sigmoid, Tanh, and Leaky ReLU
- Test each function with a range of inputs
- Verify your implementations match PyTorch's

In [1]:
# prerequisites (don't edit this block)
import torch

In [2]:
# Solution (put your code here)
def relu(x):
    """ReLU: max(0, x)"""
    return torch.clamp(x, min=0)

def sigmoid(x):
    """Sigmoid: 1 / (1 + e^(-x))"""
    return 1 / (1 + torch.exp(-x))

def tanh(x):
    """Tanh: (e^x - e^(-x)) / (e^x + e^(-x))"""
    return (torch.exp(x) - torch.exp(-x)) / (torch.exp(x) + torch.exp(-x))

def leaky_relu(x, alpha=0.01):
    """Leaky ReLU: max(alpha*x, x)"""
    return torch.max(alpha * x, x)


print(relu(torch.tensor(-3.0)))
print(torch.relu(torch.tensor(-3.0)))

print(sigmoid(torch.tensor(-3.0)))
print(torch.sigmoid(torch.tensor(-3.0)))

print(tanh(torch.tensor(-3.0)))
print(torch.tanh(torch.tensor(-3.0)))

print(leaky_relu(torch.tensor(-3.0)))
print(torch.nn.functional.leaky_relu(torch.tensor(-3.0), negative_slope=0.01))

tensor(0.)
tensor(0.)
tensor(0.0474)
tensor(0.0474)
tensor(-0.9951)
tensor(-0.9951)
tensor(-0.0300)
tensor(-0.0300)


In [3]:
# Test case (don't edit this block)
x = torch.tensor([-3.0, -1.5, -0.5, 0.0, 0.5, 1.5, 3.0])

assert torch.allclose(relu(x), torch.relu(x))
assert torch.allclose(sigmoid(x), torch.sigmoid(x))
assert torch.allclose(tanh(x), torch.tanh(x))
assert torch.allclose(leaky_relu(x), torch.nn.functional.leaky_relu(x, negative_slope=0.01))

## Task 2: Build Logic Gates with Neurons

Implement AND, OR, NAND, and NOR, gates using single neurons with manual weights.

**Tasks**:
- Design a 2-layer network with manual weights
- Test all 4 input combinations
- Explain how each hidden neuron contributes

In [4]:
# prerequisites (don't edit this block)
import torch.nn as nn

In [5]:
# Solution (put your code here)
class LogicGate(nn.Module):
    def __init__(self, gate_type):
        super().__init__()
        self.fc_hidden = nn.Linear(2, 2)
        self.fc_output = nn.Linear(2, 1)
        self.gate_type = gate_type

        with torch.no_grad():
            if gate_type == 'AND':
                self.fc_hidden.weight.copy_(torch.tensor([[1.0, 1.0], [0.0, 0.0]]))
                self.fc_hidden.bias.copy_(torch.tensor([-1.5, 0.0]))
                self.fc_output.weight.copy_(torch.tensor([[1.0, 0.0]]))
                self.fc_output.bias.copy_(torch.tensor([0.0]))
            elif gate_type == 'OR':
                self.fc_hidden.weight.copy_(torch.tensor([[1.0, 1.0], [0.0, 0.0]]))
                self.fc_hidden.bias.copy_(torch.tensor([-0.5, 0.0]))
                self.fc_output.weight.copy_(torch.tensor([[1.0, 0.0]]))
                self.fc_output.bias.copy_(torch.tensor([0.0]))
            elif gate_type == 'NAND':
                self.fc_hidden.weight.copy_(torch.tensor([[-1.0, -1.0], [0.0, 0.0]]))
                self.fc_hidden.bias.copy_(torch.tensor([1.5, 0.0]))
                self.fc_output.weight.copy_(torch.tensor([[1.0, 0.0]]))
                self.fc_output.bias.copy_(torch.tensor([0.0]))
            elif gate_type == 'NOR':
                self.fc_hidden.weight.copy_(torch.tensor([[-1.0, -1.0], [0.0, 0.0]]))
                self.fc_hidden.bias.copy_(torch.tensor([0.5, 0.0]))
                self.fc_output.weight.copy_(torch.tensor([[1.0, 0.0]]))
                self.fc_output.bias.copy_(torch.tensor([0.0]))

            for param in self.parameters():
                param.requires_grad = False

    def forward(self, x):
        y = nn.functional.relu(self.fc_hidden(x))
        y = self.fc_output(y)
        return y

In [6]:
# Test case (don't edit this block)
INPUTS = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])


def _gate_outputs(gate_type):
    gate = LogicGate(gate_type)
    gate.eval()
    with torch.no_grad():
        outputs = gate(INPUTS)
    return (outputs > 0.5).float().squeeze()


def test_and_gate():
    expected = torch.tensor([0.0, 0.0, 0.0, 1.0])
    result = _gate_outputs('AND')
    assert torch.equal(result, expected), f"AND gate failed: {result}"


def test_or_gate():
    expected = torch.tensor([0.0, 1.0, 1.0, 1.0])
    result = _gate_outputs('OR')
    assert torch.equal(result, expected), f"OR gate failed: {result}"


def test_nand_gate():
    expected = torch.tensor([1.0, 1.0, 1.0, 0.0])
    result = _gate_outputs('NAND')
    assert torch.equal(result, expected), f"NAND gate failed: {result}"


def test_nor_gate():
    expected = torch.tensor([1.0, 0.0, 0.0, 0.0])
    result = _gate_outputs('NOR')
    assert torch.equal(result, expected), f"NOR gate failed: {result}"


def test_logic_gate_has_frozen_weights():
    for gate_type in ['AND', 'OR', 'NAND', 'NOR']:
        gate = LogicGate(gate_type)
        for param in gate.parameters():
            assert not param.requires_grad, f"{gate_type} gate has unfrozen params"


def test_logic_gate_invalid_type_has_no_error():
    # Constructing with unknown type should not crash but weights are random
    gate = LogicGate('XOR')
    assert isinstance(gate, nn.Module)

## Task 3: Build XOR with Manual Weights 

Design a network that computes XOR using manually chosen weights, inspired by the absolute value MLP from the lecture.

| A | B | XOR |
|---|---|-----|
| 0 | 0 |  0  |
| 0 | 1 |  1  |
| 1 | 0 |  1  |
| 1 | 1 |  0  |

Strategy: $XOR = (A \text{ OR } B) \text{ AND NOT}(A \text{ AND } B) = (A \text{ OR } B) \text{ AND } (A \text{ NAND } B)$

**Tasks**:
- Design a 2-layer network with manual weights
- Test all 4 input combinations
- Explain how each hidden neuron contributes

In [ ]:
# Solution (put your code here)
class LogicGateXOR(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 2) # A OR B = E1, A NAND B = E2
        self.output = nn.Linear(2, 1) # E1 AND E2

        with torch.no_grad():
            pass  # TODO
            # A OR B | A NAND B
            self.hidden.weight.copy_(torch.tensor([[1.0, -1.0], [-1.0, 1.0]]))
            self.hidden.bias.copy_(torch.tensor([0.0, 0.0]))

            # E1 AND E2
            self.output.weight.copy_(torch.tensor([[1.0, 1.0]]))
            self.output.bias.copy_(torch.tensor([0.0]))

        for param in self.parameters():
            param.requires_grad = False

    def forward(self, x):
        y = nn.functional.relu(self.hidden(x))
        y = self.output(y)
        return y


# --- Der Aufruf für das XOR-Gatter ---
model_xor = LogicGateXOR()
inputs = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
 
print(f"Testlauf für Gatter: XOR")
print("-" * 40)
 
for x in inputs:
    # Wir rufen das Modell auf.
    with torch.no_grad():
        output = model_xor(x)
   
    prediction = 1 if output.item() >= 0.5 else 0
    print(f"Input: {x.tolist()} | Gatter: XOR | Output: {prediction} (Rohwert: {output.item():.4f})")

Testlauf für Gatter: XOR
----------------------------------------
Input: [0.0, 0.0] | Gatter: XOR | Output: 0 (Rohwert: 0.0000)
Input: [0.0, 1.0] | Gatter: XOR | Output: 1 (Rohwert: 1.0000)
Input: [1.0, 0.0] | Gatter: XOR | Output: 1 (Rohwert: 1.0000)
Input: [1.0, 1.0] | Gatter: XOR | Output: 0 (Rohwert: 0.0000)


In [13]:
# Test case (don't edit this block)
INPUTS = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])

def test_xor_gate():
    expected = torch.tensor([0.0, 1.0, 1.0, 0.0])
    result = _gate_outputs('XOR')
    assert torch.equal(result, expected), f"XOR gate failed: {result}"

## Task 4: Manual Forward Pass Through Multiple Neurons

Given a network with manually set weights, compute outputs step-by-step.

**Network Structure**:
- Input: 2 features
- Hidden layer: 3 neurons with ReLU
- Output layer: 1 neuron with Sigmoid

**Task**:
- Manually compute the output for input [0.5, -1.0]
- Show intermediate values (weighted sums, activations)
- Implement and verify with code

```
Hidden Layer:
  Neuron 1: W = [1.0, -0.5], b = 0.2
  Neuron 2: W = [-1.0, 2.0], b = -0.3
  Neuron 3: W = [0.5, 0.5], b = 0.0

Output Layer:
  Neuron 1: W = [1.0, -0.5, 2.0], b = 0.1
```

In [ ]:
# Solution (put your code here)
class ManualNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 3) # 2 inputs, 3 hidden neurons
        self.output = nn.Linear(3, 1) # 3 hidden neurons, 1 output
        
        # Set weights manually
        with torch.no_grad():
            self.hidden.weight[0, 0] = 1.0 # [ziel neuron, input]
            self.hidden.weight[1, 0] = -1.0 # [ziel neuron, input]
            self.hidden.weight[2, 0] = 0.5 # [ziel neuron, input]

            self.hidden.weight[0, 1] = -0.5
            self.hidden.weight[1, 1] = 2.0
            self.hidden.weight[2, 1] = 0.5

            self.hidden.bias[0] = 0.2      # [ziel neuron]
            self.hidden.bias[1] = -0.3
            self.hidden.bias[2] = 0.0

            self.output.weight[0, 0] = 1.0
            self.output.weight[0, 1] = -0.5
            self.output.weight[0, 2] = 2.0
            self.output.bias[0] = 0.1

        for param in self.parameters():
            param.requires_grad = False

    def forward(self, x):
        y = torch.relu(self.hiden(x))
        y = torch.sigmoid(self.output(y))
        return y

In [10]:
# Test case (don't edit this block)


---

Lecture: AI I - Advanced 

Next: [**Chapter 1.2: Multilayer Perceptron**](../02_mlp.ipynb)